In [3]:
import operator
from typing import List
from langgraph.graph.message import add_messages
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph,START, END, MessagesState

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, AnyMessage, ToolMessage
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = ChatGroq(model="qwen/qwen3-32b")
llm.invoke("What is the length of wall of china?")

AIMessage(content='<think>\nOkay, so I need to figure out the length of the Great Wall of China. Hmm, I remember that the Great Wall is a famous landmark, but I\'m not exactly sure how long it is. Let me start by recalling what I know.\n\nFirst, the Great Wall is a series of walls and fortifications built over different dynasties. The most well-known parts were built during the Ming Dynasty, but there were earlier sections too. I think the total length is more than a thousand miles, but I\'m not certain. Wait, I\'ve heard numbers like 13,000 miles or 14,000 miles. But is that accurate? Maybe I should break it down.\n\nI should check if there\'s a distinction between different parts of the wall. For example, the Ming Dynasty section versus older sections. Also, some parts might be better preserved than others. How is the length measured? Are they counting all the different branches and connections, or just the main line? Maybe there\'s confusion between the "Great Wall" as a single cont

# ***Develop a React Agent***

In [4]:
from langchain.tools import tool

## Custom Tool

In [16]:
@tool
def multiply(a: int, b: int) -> int:
    """Multiply tow integer number
    
    Args:
        a (int): the first integer
        b (int): the second integer
    
    """
    return a * b

@tool
def add(a: int, b: int) -> int:
    """Add two number
    
    Args:
        a (int): the first integer
        b (int): the second integer
    
    """
    return a + b

def divide(a: int, b: int) -> float:
    """Divide two number and return a float
    
    Args:
        a (int): the first integer
        b (int): the second integer
    
    """
    pass

## build in tools - Web Search

In [10]:
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

from langchain_tavily import TavilySearch

search = TavilySearch(
    max_results=3,
    topic="general"
)

In [12]:
response = search.invoke("Capital of bangladesh?")
response

{'query': 'Capital of bangladesh?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'title': 'Dhaka - Wikipedia',
   'url': 'https://en.wikipedia.org/wiki/Dhaka',
   'content': '| [1] | | Region | Southern Asia | | Notability | The Representative List of the "Intangible Cultural Heritage of Humanity" category | | | | | Dhaka (/ˈ d ɑː k ə/DAH-kə or /ˈ d æ k ə/DAK-ə; Bengali: ঢাকা, romanized:Ḍhākā, pronounced[ˈɖʱaka]ⓘ), formerly known asDacca,[20] is the capital and largest city of Bangladesh. It is one of the largest[a] and most densely populated cities[b] in the world with a density of about 34,000 people per square kilometer within a total area of approximately 300 square kilometers. [25][26] Dhaka is an important cultural, economic, and scientific hub of Eastern South Asia, as well as a major Muslim-majority city. Dhaka is also the largest Bengali-speaking city in the world.',
   'score': 0.82694983,
   'raw_content': None},
  {'title': 'Dhaka - Simple Eng

In [14]:
response['results'][2]['content']

'Bangladesh, country of South Asia, located in the delta of the Padma (Ganges [Ganga]) and Jamuna (Brahmaputra) rivers in the northeastern part of the Indian subcontinent. In the northeast and southeast—in the Sylhet and Chittagong Hills areas, respectively—the alluvial plains give place to ridges, running mainly north-south, that form part of the mountains that separate Bangladesh from Myanmar and India. In east-central Bangladesh the Brahmaputra River in its old course (the Old Brahmaputra River) built up the flood basin of the Meghna River, the region that includes the low and fertile Meghna-Sitalakhya Doab (the land area between those rivers). Lying to the south of the Feni River in southeastern Bangladesh is the Chittagong region, which has many hills, hillocks, valleys, and forests and is quite different in aspect from other parts of the country.'

In [17]:
tools = [multiply, add, divide, search]

In [18]:
llm_with_tools = llm.bind_tools(tools)

In [19]:
llm_with_tools.invoke("tell me the current political status of bangladesh")

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'bpr2s6qsb', 'function': {'arguments': '{"query":"current political status of Bangladesh","search_depth":"advanced","time_range":"week","topic":"news"}', 'name': 'tavily_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 511, 'prompt_tokens': 1248, 'total_tokens': 1759, 'completion_time': 1.029594517, 'prompt_time': 0.086323849, 'queue_time': 0.05482588000000001, 'total_time': 1.115918366}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_8487b76fd3', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--78e48c6a-3f16-4159-a004-44d9adf56fa0-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'current political status of Bangladesh', 'search_depth': 'advanced', 'time_range': 'week', 'topic': 'news'}, 'id': 'bpr2s6qsb', 'type': 'tool_call'}], usage_metadata={'input_tokens': 1248, 'output_tokens': 511, 'total_tokens': 1759})